# interactive dashboard

In [1]:
!pip install dash==2.9.3 jupyter-dash pandas plotly pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 53.0 MB/s eta 0:00:00


In [2]:
from jupyter_dash import JupyterDash
JupyterDash.infer_jupyter_proxy_config()

In [3]:

!pip install --quiet jupyter-dash plotly dash pandas

In [4]:
# Advanced interactive dashboard (Level 2) - JupyterDash
import io
import base64
import pandas as pd
import numpy as np
import plotly.express as px
from jupyter_dash import JupyterDash
from dash import Dash, dcc, html, Input, Output, State, dash_table, ctx

# ---------------------------
# Load initial data (adjust path if needed)
# ---------------------------
DEFAULT_PATH = "/content/final_crop_dataset_clean.csv"
try:
    df = pd.read_csv(DEFAULT_PATH)
except Exception:
    # fallback: small example dataset
    df = pd.DataFrame({
        "lon": np.random.uniform(34.5, 35.5, 200),
        "lat": np.random.uniform(31.5, 32.5, 200),
        "bio1": np.random.normal(250, 50, 200),
        "bio12": np.random.normal(800, 200, 200),
        "ph": np.random.randint(4,9,200),
        "sand": np.random.randint(10,90,200),
        "silt": np.random.randint(5,60,200),
        "clay": np.random.randint(5,60,200),
        "soc": np.round(np.random.uniform(0.1,5.0,200),2),
        "best_crop": np.random.choice(["wheat","maize","barley","cotton"], 200)
    })

num_cols = ["bio1","bio12","ph","sand","silt","clay","soc"]

# ---------------------------
# App layout
# ---------------------------
app = JupyterDash(__name__)
app.layout = html.Div(style={"fontFamily":"Arial, sans-serif", "padding":"12px"}, children=[

    # header
    html.Div([
        html.H2("Crop Recommendation — Interactive Dashboard", style={"margin":"0"}),
        html.Div("Interactive · Map · EDA · Upload data · Download results", style={"color":"gray","fontSize":"13px"})
    ]),

    html.Hr(),

    # Controls + Cards row
    html.Div([
        # Controls (left)
        html.Div([
            html.Label("Select / Search Crop"),
            dcc.Dropdown(
                id="crop_selector",
                options=[{"label": c, "value": c} for c in sorted(df["best_crop"].unique())],
                value=None,
                multi=True,
                placeholder="All crops (default)"
            ),

            html.Br(),
            html.Label("bio1 Range"),
            dcc.RangeSlider(
                id="bio1_range",
                min=float(np.nanmin(df["bio1"])),
                max=float(np.nanmax(df["bio1"])),
                step=(float(np.nanmax(df["bio1"]) - np.nanmin(df["bio1"])) / 100),
                value=[float(np.nanmin(df["bio1"])), float(np.nanmax(df["bio1"]))],
                tooltip={"placement":"bottom", "always_visible":False}
            ),

            html.Br(),
            html.Label("pH Range"),
            dcc.RangeSlider(
                id="ph_range",
                min=int(np.nanmin(df["ph"])),
                max=int(np.nanmax(df["ph"])),
                step=1,
                value=[int(np.nanmin(df["ph"])), int(np.nanmax(df["ph"]))],
                marks={i: str(i) for i in range(int(np.nanmin(df["ph"])), int(np.nanmax(df["ph"]))+1)}
            ),

            html.Br(),
            html.Label("Select Axes for Plot (X / Y)"),
            dcc.Dropdown(id="x_axis", options=[{"label":c, "value":c} for c in num_cols], value="sand", clearable=False),
            dcc.Dropdown(id="y_axis", options=[{"label":c, "value":c} for c in num_cols], value="clay", clearable=False),

            html.Br(),
            html.Label("Color by"),
            dcc.RadioItems(id="color_by", options=[
                {"label":"best_crop", "value":"best_crop"},
                {"label":"ph", "value":"ph"},
                {"label":"bio1", "value":"bio1"}
            ], value="best_crop", inline=True),

            html.Br(),
            html.Label("Upload CSV (optional)"),
            dcc.Upload(
                id="upload-data",
                children=html.Div(["Drag and Drop or Click to Select (.csv)"]),
                style={
                    'width': '100%', 'height': '60px', 'lineHeight': '60px',
                    'borderWidth': '1px', 'borderStyle': 'dashed',
                    'borderRadius': '4px', 'textAlign': 'center', 'marginBottom': '8px'
                },
                multiple=False
            ),

            html.Button("Download filtered CSV", id="download-btn"),
            dcc.Download(id="download-dataframe-csv")
        ], style={"width":"28%", "display":"inline-block", "verticalAlign":"top", "paddingRight":"12px"}),

        # Cards (right)
        html.Div([
            html.Div(id="cards_row", style={"display":"flex", "gap":"10px", "flexWrap":"wrap"})
        ], style={"width":"70%", "display":"inline-block", "verticalAlign":"top"})
    ], style={"marginBottom":"18px"}),

    # Tabs
    dcc.Tabs(id="tabs", value="tab-overview", children=[
        dcc.Tab(label="Overview", value="tab-overview", children=[
            html.Div([
                html.Div([dcc.Graph(id="map")], style={"width":"49%", "display":"inline-block"}),
                html.Div([dcc.Graph(id="scatter")], style={"width":"49%", "display":"inline-block"})
            ]),
            html.Div([
                html.Div([dcc.Graph(id="hist")], style={"width":"49%", "display":"inline-block"}),
                html.Div([dcc.Graph(id="corr")], style={"width":"49%", "display":"inline-block"})
            ])
        ]),
        dcc.Tab(label="Map", value="tab-map", children=[
            html.Div([dcc.Graph(id="map_only", style={"height":"700px"})])
        ]),
        dcc.Tab(label="EDA", value="tab-eda", children=[
            html.Div([
                html.H4("Distribution & Boxplots"),
                html.Div([dcc.Graph(id="multi_hist")], style={"width":"100%"}),
            ])
        ]),
        dcc.Tab(label="Table", value="tab-table", children=[
            html.H4("Data Table"),
            dash_table.DataTable(
                id="data_table",
                columns=[{"name":c, "id":c} for c in df.columns],
                page_size=12,
                filter_action="native",
                sort_action="native",
                style_table={"overflowX":"auto"},
            )
        ])
    ])
])

# ---------------------------
# Helper: parse uploaded CSV
# ---------------------------
def parse_contents(contents):
    content_type, content_string = contents.split(',')
    decoded = base64.b64decode(content_string)
    try:
        return pd.read_csv(io.StringIO(decoded.decode('utf-8')))
    except Exception as e:
        try:
            return pd.read_excel(io.BytesIO(decoded))
        except Exception as e2:
            raise ValueError("Failed to read file, make sure it is CSV or Excel.") from e2

# ---------------------------
# Callbacks
# ---------------------------

# update data when upload occurs (store in-memory by returning to callbacks through State)
@app.callback(
    Output("crop_selector", "options"),
    Output("crop_selector", "value"),
    Output("x_axis", "options"),
    Output("y_axis", "options"),
    Output("data_table", "columns"),
    Input("upload-data", "contents"),
    State("upload-data", "filename")
)
def update_on_upload(contents, filename):
    global df
    if contents:
        try:
            newdf = parse_contents(contents)
            df = newdf.copy()
        except Exception as e:
            print("Upload error:", e)
    # prepare controls
    crop_opts = [{"label": c, "value": c} for c in sorted(df["best_crop"].unique())] if "best_crop" in df.columns else []
    xopts = [{"label":c, "value":c} for c in [c for c in df.columns if df[c].dtype in [np.float64, np.int64]]]
    yopts = xopts
    columns = [{"name":c, "id":c} for c in df.columns]
    return crop_opts, None, xopts, yopts, columns

# Filtered dataframe generator
def filter_df(selected_crops, bio1_rng, ph_rng):
    dff = df.copy()
    # filters
    if selected_crops and len(selected_crops)>0:
        if "best_crop" in dff.columns:
            dff = dff[dff["best_crop"].isin(selected_crops)]
    if "bio1" in dff.columns:
        dff = dff[(dff["bio1"] >= bio1_rng[0]) & (dff["bio1"] <= bio1_rng[1])]
    if "ph" in dff.columns:
        dff = dff[(dff["ph"] >= ph_rng[0]) & (dff["ph"] <= ph_rng[1])]
    return dff

# Update cards row & graphs
@app.callback(
    Output("cards_row", "children"),
    Output("map", "figure"),
    Output("map_only", "figure"),
    Output("scatter", "figure"),
    Output("hist", "figure"),
    Output("corr", "figure"),
    Output("multi_hist", "figure"),
    Output("data_table", "data"),
    Input("crop_selector", "value"),
    Input("bio1_range", "value"),
    Input("ph_range", "value"),
    Input("x_axis", "value"),
    Input("y_axis", "value"),
    Input("color_by", "value")
)
def update_dashboard(crops, bio1_rng, ph_rng, xcol, ycol, color_by):
    dff = filter_df(crops, bio1_rng, ph_rng)

    # Cards
    total = len(dff)
    unique_crops = dff["best_crop"].nunique() if "best_crop" in dff.columns else 0
    mean_ph = round(float(dff["ph"].mean()),2) if "ph" in dff.columns else None
    mean_bio1 = round(float(dff["bio1"].mean()),2) if "bio1" in dff.columns else None

    cards = [
        html.Div([html.H6("Total records", style={"margin":"0","color":"#555"}),
                  html.Div(f"{total}", style={"fontSize":"20px","fontWeight":"700"})],
                 style={"padding":"10px","background":"#f7f7f7","borderRadius":"6px","width":"150px","textAlign":"center"}),
        html.Div([html.H6("Unique crops", style={"margin":"0","color":"#555"}),
                  html.Div(f"{unique_crops}", style={"fontSize":"20px","fontWeight":"700"})],
                 style={"padding":"10px","background":"#f7f7f7","borderRadius":"6px","width":"150px","textAlign":"center"}),
        html.Div([html.H6("Mean pH", style={"margin":"0","color":"#555"}),
                  html.Div(f"{mean_ph}", style={"fontSize":"20px","fontWeight":"700"})],
                 style={"padding":"10px","background":"#f7f7f7","borderRadius":"6px","width":"150px","textAlign":"center"}),
        html.Div([html.H6("Mean bio1", style={"margin":"0","color":"#555"}),
                  html.Div(f"{mean_bio1}", style={"fontSize":"20px","fontWeight":"700"})],
                 style={"padding":"10px","background":"#f7f7f7","borderRadius":"6px","width":"150px","textAlign":"center"})
    ]

    # Map
    if {"lon","lat"}.issubset(dff.columns):
        map_fig = px.scatter_mapbox(
            dff, lat="lat", lon="lon", color=color_by if color_by in dff.columns else None,
            hover_data=[c for c in dff.columns if c not in ["lon","lat"]],
            zoom=5, height=450
        )
        map_fig.update_layout(mapbox_style="open-street-map", margin={"r":0,"t":30,"l":0,"b":0})
        map_fig.update_traces(marker={"size":8, "opacity":0.8})
        map_only_fig = map_fig
    else:
        map_fig = px.scatter(title="No lon/lat columns in dataset")
        map_only_fig = map_fig

    # Scatter
    if xcol in dff.columns and ycol in dff.columns:
        scatter_fig = px.scatter(dff, x=xcol, y=ycol, color=color_by if color_by in dff.columns else None,
                                 hover_data=["best_crop"] if "best_crop" in dff.columns else None, trendline="ols")
    else:
        scatter_fig = px.scatter(title="Choose numeric X and Y")

    # Histogram (of selected x)
    hist_fig = px.histogram(dff, x=xcol if xcol in dff.columns else (num_cols[0] if num_cols[0] in dff.columns else None),
                            nbins=30, title=f"Distribution of {xcol}")

    # Correlation heatmap
    available_num = [c for c in num_cols if c in dff.columns]
    if len(available_num) >= 2:
        corr = dff[available_num].corr()
        corr_fig = px.imshow(corr, title="Correlation (numeric features)")
    else:
        corr_fig = px.imshow([[1]], x=["no_num"], y=["no_num"], title="Not enough numeric cols for correlation")

    # Multi histogram / box
    multi_hist = px.histogram(dff.melt(value_vars=available_num), x="value", facet_col="variable",
                              facet_col_wrap=3, title="Distributions (multiple features)")

    # Data table
    table_data = dff.to_dict("records")

    return cards, map_fig, map_only_fig, scatter_fig, hist_fig, corr_fig, multi_hist, table_data

# Download filtered CSV
@app.callback(
    Output("download-dataframe-csv", "data"),
    Input("download-btn", "n_clicks"),
    State("crop_selector", "value"),
    State("bio1_range", "value"),
    State("ph_range", "value"),
    prevent_initial_call=True
)
def download_filtered(nclicks, crops, bio1_rng, ph_rng):
    dff = filter_df(crops, bio1_rng, ph_rng)
    return dcc.send_data_frame(dff.to_csv, "filtered_crops.csv", index=False)

# Run server inline in Jupyter
if __name__ == "__main__":
    app.run_server(mode="inline")


Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



<IPython.core.display.Javascript object>